# 1 · Train one MiniFASNet model

Trains **one** model for **one patch type** (how much of the surroundings the crop around the face includes).
The service ensembles one model per patch type, so run this once per type you want (e.g. `2.7_80x80` and `4_80x80`).

**Data** — cropped patches, one folder per label (`1` = real face, `0` / `2` = attack types):
```
datasets/rgb_image/train/<patch_info>/{0,1,2}/*.png
```
**Output** — checkpoints every `save_every` steps in `saved_logs/snapshot/Anti_Spoofing_<patch_info>/`,
TensorBoard logs in `saved_logs/jobs/`. Next: **02_validate** picks the best checkpoint.

The training code lives in `src/model_development/` (trainer, dataset, loss); this notebook only sets parameters and runs it.

## Parameters — the only cell you normally edit

In [ ]:
from model_development.config import REPO_ROOT, get_default_config, update_config

PATCH_INFO = "2.7_80x80"   # org_1_80x60 | 1_80x80 | 2.7_80x80 | 4_80x80
DEVICES = [0]              # GPU ids; ignored when there is no GPU

conf = get_default_config()
conf.train_root_path = REPO_ROOT / "datasets" / "rgb_image" / "train"
conf.epochs = 25
conf.batch_size = 1024
conf.lr = 1e-1
conf.milestones = [10, 15, 22]   # epochs at which the learning rate drops by `gamma`
conf.save_every = 30             # steps between checkpoints
conf.num_workers = 4

## Check the data

In [ ]:
conf = update_config(conf, PATCH_INFO, DEVICES)
data_dir = conf.train_root_path / PATCH_INFO
counts = {d.name: sum(1 for _ in d.iterdir()) for d in sorted(data_dir.iterdir()) if d.is_dir()}
print("training data:", data_dir)
print("images per label (1 = real):", counts)
print("device:", conf.device, "| input:", conf.input_size, "| checkpoints ->", conf.model_path)
assert set(counts) == {"0", "1", "2"}, "expected label folders 0, 1 and 2"

## Train

In [ ]:
from model_development.trainer import TrainMain

TrainMain(conf).train_model()

## Checkpoints written

In [ ]:
for path in sorted(conf.model_path.glob("*.pth")):
    print(path.name)